In [ ]:
#import libraries

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
#dataset path and loading metadata

dataset_path = "desktop/nna272/nayla/msc_thesis/breast_cancer/DMID_dataset"

csv_path = os.path.join(dataset_path, "metadata.csv")

df = pd.read_csv(csv_path)

print(df.head())
print(df.shape)

In [ ]:
#removing null and duplicate values

df = df.dropna()

df = df.drop_duplicates()

print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
#encoding labels and creating image paths

label_map = {
    'negative':0,
    'positive':1
}

df['label'] = df['cancer'].map(label_map)

df['image_path'] = df['image_file_path'].apply(
    lambda x: os.path.join(dataset_path, x)
)

df = df[df['image_path'].apply(os.path.exists)]

print(df['label'].value_counts())

In [ ]:
#image preprocessing and train test split

IMG_SIZE = 224

images = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    img = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)

    if img is not None:

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        img = img / 255.0

        images.append(img)

        labels.append(row['label'])

X = np.array(images)
y = np.array(labels)

X = X.reshape(-1, 224, 224, 1)

y = to_categorical(y)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)